# L5: Voice AI Evals

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em> and select <em>"Notebook (.ipynb)"</em>.</p>
</div>

<br>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different run results:</b> The output generated by AI models can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from those shown in the video.</p>

**Knowing Your Voice Agent Is Actually Good**

You've built three working voice agents. Now you'll install
the discipline that separates a demo from production:
**eval-driven development**.

Vocal Bridge ships a built-in multimodal evaluator. Give it a
recorded session ID and an objective; it sends the audio,
the agent's full config, the transcript, and the tool-call
log to a multimodal LLM, which scores the call and proposes
concrete prompt improvements.

### In this lesson you'll:

1. Run a single eval against a real call recording.
2. Read the structured score and suggestions.
3. Build a tiny eval suite (multiple sessions, one
   objective each).
4. Walk the iterate-loop: identify a failure → patch the
   prompt → re-eval → confirm.


## Setup

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note:</b> You need at least one **completed call** with a recording
attached. The easiest source: the call you placed in
Lesson 4 — pick its `session_id` below.</p>


In [ ]:
import json
import os
from pathlib import Path
from helpers import load_env, vb, eval_session

env = load_env()
assert os.getenv("VOCAL_BRIDGE_API_KEY"), "Set VOCAL_BRIDGE_API_KEY in .env"
print("API key loaded ✓")

### What's in `helpers.py`

| Helper | What it does |
| --- | --- |
| `vb(*args)` | Runs a `vb` CLI command |
| `eval_session(session_id, objective)` | Runs `vb eval` and returns the report dict |


## Step 1 — Find a session to evaluate

Pull the most recent completed calls so you can pick one.


In [ ]:
recent = vb(
    "logs", "list", "-n", "5",
    "--status", "completed",
    json_output=True,
)
sessions = recent.get("logs") or recent.get("sessions") or []
for r in sessions:
    sid = r.get("session_id") or r.get("id")
    started = r.get("started_at", "?")
    dur = r.get("duration_seconds", "?")
    print(f"  {sid}  {started}  {dur}s")


## Step 2 — Evaluate one call against an objective

Pick a `session_id` from above and an objective the agent
was supposed to accomplish and paste it in the `SESSION_ID` and `OBJECTIVE` variables below. The objective is what makes
evals *meaningful* — without one, the LLM is grading vibes.

Under the hood, `eval_session()` is one CLI command:

```bash
vb eval <session_id> \
   --objective "Confirm the appointment for tomorrow at 2pm" \
   --json
```

The CLI sends the call audio + agent config + transcript +
tool-call log to a multimodal LLM, which scores the call
against the objective and returns concrete prompt edits.
**One command — that's the whole eval API.**


In [ ]:
SESSION_ID = "INPUT A SESSION ID FROM THE ABOVE CELL OUTPUT HERE"
OBJECTIVE = (
    "INPUT THE OBJECTIVE HERE"
)

report = eval_session(SESSION_ID, objective=OBJECTIVE)
print(json.dumps(report, indent=2)[:2000])


## Step 3 — Read the report

Eval reports come back as a structured object. The two
things you'll act on:

- **score** — a 0–10 quality rating.
- **suggestions** — concrete edits to your prompt or tool
  config.

If the call missed the objective, the model tells you
exactly *what* went wrong (e.g. *"agent didn't confirm the
purpose before hanging up"*). That's your prompt diff.


In [ ]:
print(f"Score:   {report['result'].get('score', '—')}")
print(f"Verdict: {report['result'].get('verdict', '—')}")
print()
print("Suggestions for prompt improvements:")
print(report['result'].get("suggested_prompt_improvements"))

## Step 4 — Build a small eval suite

One call isn't a suite. Pick 3–5 sessions that cover the
failure modes you care about (voicemail, busy callee,
ambiguous instructions, interruption recovery), each with an
objective + scenario.

Run them in a loop and tabulate the results.


In [ ]:
SUITE = [
    # (session_id, objective, scenario)
    # (
    #     "abc123…",
    #     "Confirm appointment for tomorrow at 2pm.",
    #     "Callee confirms.",
    # ),
    # (
    #     "def456…",
    #     "Confirm appointment for tomorrow at 2pm.",
    #     "Callee asks to reschedule.",
    # ),
]

results = []
for session_id, objective, scenario in SUITE:
    rep = eval_session(
        session_id,
        objective=objective,
        scenario=scenario,
    )
    results.append({
        "session": session_id[:8],
        "score":   rep['result'].get("score"),
        "verdict": rep['result'].get("verdict"),
    })

import pprint
pprint.pp(results)


## Step 5 — Close the loop

Pick the worst-scoring session and look at the suggestions.
Patch the prompt with `vb prompt set -f <updated.md>`, place
a new test call, then re-evaluate.

The discipline is the loop, not the single number:

```
build → call → eval → suggest → patch → call → eval → ✓
```

A regression isn't fixed until a fresh call against the
**same objective** scores higher. The eval suite is what
keeps you honest.


In [ ]:
# Example patch loop — uncomment and adapt to your session.

# 1. Read the current prompt
# current = vb("prompt", "show")
# print(current)

# 2. Edit it
# new_prompt = current.replace("OLD_LINE", "NEW_LINE")
# Path("/tmp/updated_prompt.md").write_text(new_prompt)

# 3. Push it
# print(vb("prompt", "set", "-f", "/tmp/updated_prompt.md"))

# 4. Place another test call (Lesson 4 pattern), capture
#    its new session_id, then re-eval against the same
#    objective.

# report2 = eval_session(NEW_SESSION_ID, objective=OBJECTIVE)
# print("before:", report.get("score"),
#       "after:",  report2.get("score"))


## What's next

You've now built across all three Vocal Bridge surfaces and
installed the eval-driven dev loop. That's the course.

From here the natural next step is hardening one of your
demos for production: auth, monitoring, latency tuning, and
the checklist that turns a working demo into a deployed
product. The Vocal Bridge docs at
[vocalbridgeai.com/docs/overview](https://vocalbridgeai.com/docs/overview) pick
up from exactly there.
